---
title: "Lab 5: The 12 Days of Christmas"
author: "Shiqi Wu"
format:
  html:
    embed-resources: true
    code-overflow: wrap
execute:
  enabled: false
---

In [1]:
import pandas as pd

xmas = pd.read_csv("https://www.dropbox.com/scl/fi/qxaslqqp5p08i1650rpc4/xmas.csv?rlkey=erdxi7jbh7pqf9fh4lv4cayp5&dl=1")

xmas

,Day,Day.in.Words,Gift.Item,Verb,Adjective,Location
0,1,first,partridge,NaN,NaN,in a pear tree
1,2,second,dove,NaN,turtle,NaN
2,3,third,hen,NaN,french,NaN
3,4,fourth,bird,NaN,calling,NaN
4,5,fifth,ring,NaN,golden,NaN
5,6,sixth,goose,a-laying,NaN,NaN
6,7,seventh,swan,a-swimming,NaN,NaN
7,8,eighth,maid,a-milking,NaN,NaN
8,9,ninth,lady,dancing,NaN,NaN
9,10,tenth,lord,a-leaping,NaN,NaN


## Function 1: pluralize_gift()

In [2]:
def pluralize_gift(gift):
    """
    Return the plural form of a gift noun.

    Parameters
    ----------
    gift : str
        A singular gift noun.

    Returns
    -------
    str
        The plural form of the noun.
    """
    if "oo" in gift:
        gift = gift.replace("oo", "ee")
    elif gift.endswith("y") and gift[-2] not in "aeiou":
        gift = gift[:-1] + "ies"
    else:
        gift = gift + "s"

    return gift

In [3]:
print(pluralize_gift("goose"))
print(pluralize_gift("lady"))
print(pluralize_gift("ring"))
print(pluralize_gift("toy"))

geese
ladies
rings
toys


In [4]:
xmas["Gift.Item"].head(2).map(pluralize_gift)

0    partridges
1         doves
Name: Gift.Item, dtype: str

In [5]:
xmas["Gift.Item"].map(pluralize_gift)

0     partridges
1          doves
2           hens
3          birds
4          rings
5          geese
6          swans
7          maids
8         ladies
9          lords
10        pipers
11      drummers
Name: Gift.Item, dtype: str

Answer: This function is not vectorized because it handles one string at a time. I used `.map(pluralize_gift)` to apply it to each gift in the column. Both the smaller and full gift datasets produced the correct plural forms.

## Function 2: make_phrase()

In [6]:
number_words = {
    1: "one", 2: "two", 3: "three", 4: "four",
    5: "five", 6: "six", 7: "seven", 8: "eight",
    9: "nine", 10: "ten", 11: "eleven", 12: "twelve"
}

xmas["Num.in.Words"] = xmas["Day"].map(number_words)

xmas[["Day", "Day.in.Words", "Num.in.Words"]]

,Day,Day.in.Words,Num.in.Words
0,1,first,one
1,2,second,two
2,3,third,three
3,4,fourth,four
4,5,fifth,five
5,6,sixth,six
6,7,seventh,seven
7,8,eighth,eight
8,9,ninth,nine
9,10,tenth,ten


In [7]:
def make_phrase(num, num_word, item, verb, adjective, location):
    """
    Build a phrase describing a gift.

    Parameters
    ----------
    num : int
        The number of gifts.
    num_word : str
        The number written in words.
    item : str
        The singular gift noun.
    verb, adjective, location : str
        Gift details, which may be missing.

    Returns
    -------
    str
        The complete gift phrase.
    """
    if pd.isna(verb):
        verb = ""
    if pd.isna(adjective):
        adjective = ""
    if pd.isna(location):
        location = ""

    if num > 1:
        item = pluralize_gift(item)
    else:
        if item[0].lower() in "aeiou":
            num_word = "an"
        else:
            num_word = "a"

    phrase = num_word + " " + adjective + " " + item + " " + verb + " " + location

    return " ".join(phrase.split())

In [8]:
print(make_phrase(10, "ten", "lord", "a-leaping", "", ""))
print(make_phrase(1, "one", "partridge", None, None, "in a pear tree"))
print(make_phrase(1, "one", "apple", None, None, None))

ten lords a-leaping
a partridge in a pear tree
an apple


In [9]:
xmas.head(2).apply(
    lambda row: make_phrase(
        row["Day"], row["Num.in.Words"], row["Gift.Item"],
        row["Verb"], row["Adjective"], row["Location"]
    ),
    axis=1
)

0    a partridge in a pear tree
1              two turtle doves
dtype: str

In [10]:
xmas["Full.Phrase"] = xmas.apply(
    lambda row: make_phrase(
        row["Day"], row["Num.in.Words"], row["Gift.Item"],
        row["Verb"], row["Adjective"], row["Location"]
    ),
    axis=1
)

xmas[["Day", "Full.Phrase"]]

,Day,Full.Phrase
0,1,a partridge in a pear tree
1,2,two turtle doves
2,3,three french hens
3,4,four calling birds
4,5,five golden rings
5,6,six geese a-laying
6,7,seven swans a-swimming
7,8,eight maids a-milking
8,9,nine ladies dancing
9,10,ten lords a-leaping


## Function 3: sing_day()

In [11]:
def sing_day(dataset, num, phrase_col):
    """
    Return the complete lyrics for one day.

    Parameters
    ----------
    dataset : pandas.DataFrame
        The gift dataset.
    num : int
        The day to sing.
    phrase_col : str
        The column containing gift phrases.

    Returns
    -------
    str
        The opening line and all gifts for that day.
    """
    num_word = dataset.loc[dataset["Day"] == num, "Day.in.Words"].iloc[0]
    intro = "On the " + num_word + " day of Christmas, my true love sent to me:"

    gifts = dataset.loc[dataset["Day"] <= num].sort_values("Day", ascending=False)[phrase_col].tolist()

    if num > 1:
        gifts[-1] = "and " + gifts[-1]

    return intro + "\n" + ",\n".join(gifts) + "."

In [12]:
print(sing_day(xmas, 3, "Full.Phrase"))

On the third day of Christmas, my true love sent to me:
three french hens,
two turtle doves,
and a partridge in a pear tree.


In [13]:
print(sing_day(xmas.head(2), 1, "Full.Phrase"))

On the first day of Christmas, my true love sent to me:
a partridge in a pear tree.


## Use Your Functions!

### The 12 Days of Christmas

In [14]:
song = xmas["Day"].map(lambda day: sing_day(xmas, day, "Full.Phrase"))

print("\n\n".join(song))

On the first day of Christmas, my true love sent to me:
a partridge in a pear tree.

On the second day of Christmas, my true love sent to me:
two turtle doves,
and a partridge in a pear tree.

On the third day of Christmas, my true love sent to me:
three french hens,
two turtle doves,
and a partridge in a pear tree.

On the fourth day of Christmas, my true love sent to me:
four calling birds,
three french hens,
two turtle doves,
and a partridge in a pear tree.

On the fifth day of Christmas, my true love sent to me:
five golden rings,
four calling birds,
three french hens,
two turtle doves,
and a partridge in a pear tree.

On the sixth day of Christmas, my true love sent to me:
six geese a-laying,
five golden rings,
four calling birds,
three french hens,
two turtle doves,
and a partridge in a pear tree.

On the seventh day of Christmas, my true love sent to me:
seven swans a-swimming,
six geese a-laying,
five golden rings,
four calling birds,
three french hens,
two turtle doves,
and a 

### Surprise Song

In [15]:
xmas2 = pd.read_csv("https://www.dropbox.com/scl/fi/p9x9k8xwuzs9rhp582vfy/xmas_2.csv?rlkey=kvc3j3lmyn4opcidsrhcmrof1&dl=1")

xmas2

,Day,Day.in.Words,Gift.Item,Verb,Adjective,Location
0,1,first,email,NaN,NaN,from Cal Poly
1,2,second,point,NaN,meal,NaN
2,3,third,pen,NaN,lost,NaN
3,4,fourth,review,NaN,course,NaN
4,5,fifth,exam,NaN,practice,NaN
5,6,sixth,grader,grading,NaN,NaN
6,7,seventh,senior,stressing,NaN,NaN
7,8,eighth,mom,a-calling,NaN,NaN
8,9,ninth,party,bumping,NaN,NaN
9,10,tenth,load,of laundry,NaN,NaN


In [16]:
xmas2["Num.in.Words"] = xmas2["Day"].map(number_words)

xmas2[["Day", "Day.in.Words", "Num.in.Words"]]

,Day,Day.in.Words,Num.in.Words
0,1,first,one
1,2,second,two
2,3,third,three
3,4,fourth,four
4,5,fifth,five
5,6,sixth,six
6,7,seventh,seven
7,8,eighth,eight
8,9,ninth,nine
9,10,tenth,ten


In [17]:
xmas2["Full.Phrase"] = xmas2.apply(
    lambda row: make_phrase(
        row["Day"], row["Num.in.Words"], row["Gift.Item"],
        row["Verb"], row["Adjective"], row["Location"]
    ),
    axis=1
)

xmas2[["Day", "Full.Phrase"]]

,Day,Full.Phrase
0,1,an email from Cal Poly
1,2,two meal points
2,3,three lost pens
3,4,four course reviews
4,5,five practice exams
5,6,six graders grading
6,7,seven seniors stressing
7,8,eight moms a-calling
8,9,nine parties bumping
9,10,ten loads of laundry


In [18]:
song2 = xmas2["Day"].map(lambda day: sing_day(xmas2, day, "Full.Phrase"))

print("\n\n".join(song2))

On the first day of Christmas, my true love sent to me:
an email from Cal Poly.

On the second day of Christmas, my true love sent to me:
two meal points,
and an email from Cal Poly.

On the third day of Christmas, my true love sent to me:
three lost pens,
two meal points,
and an email from Cal Poly.

On the fourth day of Christmas, my true love sent to me:
four course reviews,
three lost pens,
two meal points,
and an email from Cal Poly.

On the fifth day of Christmas, my true love sent to me:
five practice exams,
four course reviews,
three lost pens,
two meal points,
and an email from Cal Poly.

On the sixth day of Christmas, my true love sent to me:
six graders grading,
five practice exams,
four course reviews,
three lost pens,
two meal points,
and an email from Cal Poly.

On the seventh day of Christmas, my true love sent to me:
seven seniors stressing,
six graders grading,
five practice exams,
four course reviews,
three lost pens,
two meal points,
and an email from Cal Poly.

On t